This notebook corresponds to the figures in the supplementary info, specifically in choosing the parameters to generate Burr distributions, and ablation studies.

In [ ]:
import os
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
for candidate in [PROJECT_ROOT, *PROJECT_ROOT.parents]:
    if (candidate / "network_generator").exists():
        PROJECT_ROOT = candidate
        break
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)

import gc
import torch
import torch.nn as nn
from tqdm import tqdm
import numpy as np
import pandas as pd
import igraph as ig
import matplotlib.pyplot as plt
from network_generator.config import NetworkConfig, TargetConfig
from network_generator.initialization import (
    directed_configuration_model,
    generate_burr_degrees,
    generate_burr_degrees_matching_sum,
    get_parameters_for_burr_distribution,
    triangular_wedge_matching,
)
from network_generator.metrics import (
    compute_degrees_from_model,
    compute_hill_exponent,
)

criterion_sse = nn.MSELoss(reduction="sum")
criterion_mse = nn.MSELoss()

In [ ]:
# Configuration - modify these config classes to change parameters
network_config = NetworkConfig(num_firms=100_000)
target_config = TargetConfig()

# For backward compatibility, extract values to individual variables
num_firms = network_config.num_firms
monetary_unit = network_config.monetary_unit
country_code = network_config.country_code
country_year = network_config.country_year

In [ ]:
# Extract target values from config for backward compatibility
target_hill_in_deg = torch.tensor(target_config.target_hill_in_deg, dtype=torch.float32)
target_hill_out_deg = torch.tensor(target_config.target_hill_out_deg, dtype=torch.float32)
target_hill_weight_distribution = torch.tensor(target_config.target_hill_weight_distribution, dtype=torch.float32)
target_hill_in_str = torch.tensor(target_config.target_hill_in_str, dtype=torch.float32)
target_hill_out_str = torch.tensor(target_config.target_hill_out_str, dtype=torch.float32)
target_average_connection = torch.tensor(target_config.target_average_connection, dtype=torch.float32)

target_so_ko_corr = torch.tensor(target_config.target_so_ko_corr, dtype=torch.float32)
target_si_ki_corr = torch.tensor(target_config.target_si_ki_corr, dtype=torch.float32)
target_si_so_corr = torch.tensor(target_config.target_si_so_corr, dtype=torch.float32)
target_ki_ko_corr = torch.tensor(target_config.target_ki_ko_corr, dtype=torch.float32)

target_so_ko_regcoeff = torch.tensor(target_config.target_so_ko_regcoeff, dtype=torch.float32)
target_ko_so_regcoeff = torch.tensor(target_config.target_ko_so_regcoeff, dtype=torch.float32)
target_si_ki_regcoeff = torch.tensor(target_config.target_si_ki_regcoeff, dtype=torch.float32)
target_ki_si_regcoeff = torch.tensor(target_config.target_ki_si_regcoeff, dtype=torch.float32)

target_tls_str = torch.tensor(target_config.target_tls_str, dtype=torch.float32)
target_tls_deg = torch.tensor(target_config.target_tls_deg, dtype=torch.float32)

burr_max_indeg = torch.tensor(network_config.burr_max_indeg, dtype=torch.float32)
burr_max_outdeg = torch.tensor(network_config.burr_max_outdeg, dtype=torch.float32)

# Step 1: Estimate $\alpha,\nu$ for the out- and in-degrees at hill001 from engineered Burr distributions.

In [ ]:
from scipy.stats import burr12
alpha_grid = np.arange(1.5, 7.5, 0.5)
nu_grid = np.arange(1, 11.5, 0.5)
target_average_connection = torch.tensor([40], dtype = torch.float64)
hill_values = np.empty((len(alpha_grid), len(nu_grid)))
logvariance_values = np.empty((len(alpha_grid), len(nu_grid)))

num_reps_out = 100

total_iters = len(alpha_grid) * len(nu_grid)

with tqdm(total=total_iters, desc="Computing Hill surface") as pbar:
    for i, alpha_val in enumerate(alpha_grid):
        for j, nu_val in enumerate(nu_grid):
            k_outdeg, c_outdeg, lam_outdeg = get_parameters_for_burr_distribution(
                target_alpha_input=torch.tensor([alpha_val], dtype=torch.float64),
                target_log_var_input=torch.tensor([nu_val], dtype=torch.float64),
                target_mean_input=target_average_connection
            )

            running_sum_hill = 0.0
            running_sum_logvariance = 0.0
            for _ in range(num_reps_out):
                sample_torch = generate_burr_degrees(c_val= c_outdeg,
                                             d_val= k_outdeg,
                                             lam_val= lam_outdeg,
                                             max_cutoff= burr_max_outdeg,
                                             num_of_firms=100_000,
                                            )

                running_sum_hill += compute_hill_exponent(
                    sample_torch,
                    network_config.threshold_percentile_for_hill_deg,
                    is_degree=False
                ).item()

                running_sum_logvariance += sample_torch[sample_torch > 0].log().var().item()
            hill_values[i, j] = running_sum_hill / num_reps_out
            logvariance_values[i, j] = running_sum_logvariance / num_reps_out
            pbar.update(1)

In [ ]:
Alpha, Nu = np.meshgrid(alpha_grid, nu_grid, indexing="ij")

plt.figure(figsize=(16, 8))

pcm = plt.pcolormesh(Nu, Alpha, hill_values, shading="auto")
plt.xlabel(r'user-specified log-variance value in BurrXII, $\nu$', fontsize = 14)
plt.ylabel(r'user-specified tail exponent value in BurrXII, $\alpha = ck$', fontsize = 14)

cbar = plt.colorbar(pcm, pad = 0.01)
cbar.set_label('Estimated Hill exponent')

# --- Add text annotations ---
for i in range(len(alpha_grid)):
    for j in range(len(nu_grid)):
        plt.text(
            nu_grid[j],                 # x (nu)
            alpha_grid[i],              # y (alpha)
            f"({logvariance_values[i, j]:.2f}, \n{hill_values[i, j]:.2f})",
            ha='center',
            va='center',
            fontsize=10,
            color='white'
        )
plt.title(f"Estimated (log-variance, Hill exponent) of Burr out-degrees averaged over {num_reps_out} experiments",fontsize=16)
plt.xticks(fontsize=12)
plt.yticks(fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
from scipy.stats import burr12
alpha_grid = np.arange(1.5, 9.5, 0.5)
nu_grid = np.arange(1, 7.5, 0.5)

hill_values = np.empty((len(alpha_grid), len(nu_grid)))
logvariance_values = np.empty((len(alpha_grid), len(nu_grid)))

num_reps_in = 100

total_iters = len(alpha_grid) * len(nu_grid)

with tqdm(total=total_iters, desc="Computing Hill surface") as pbar:
    for i, alpha_val in enumerate(alpha_grid):
        for j, nu_val in enumerate(nu_grid):
            k_indeg, c_indeg, lam_indeg = get_parameters_for_burr_distribution(
                target_alpha_input=torch.tensor([alpha_val], dtype=torch.float64),
                target_log_var_input=torch.tensor([nu_val], dtype=torch.float64),
                target_mean_input=target_average_connection
            )

            running_sum_hill = 0.0
            running_sum_logvariance = 0.0
            for _ in range(num_reps_in):
                sample_torch = generate_burr_degrees(c_val= c_indeg,
                                             d_val= k_indeg,
                                             lam_val= lam_indeg,
                                             max_cutoff= burr_max_indeg,
                                             num_of_firms=100_000,
                                            )

                running_sum_hill += compute_hill_exponent(
                    sample_torch,
                    network_config.threshold_percentile_for_hill_deg,
                    is_degree=False
                ).item()

                running_sum_logvariance += sample_torch[sample_torch > 0].log().var().item()
            hill_values[i, j] = running_sum_hill / num_reps_in
            logvariance_values[i, j] = running_sum_logvariance / num_reps_in
            pbar.update(1)

In [ ]:
Alpha, Nu = np.meshgrid(alpha_grid, nu_grid, indexing="ij")

plt.figure(figsize=(14,11))

pcm = plt.pcolormesh(Nu, Alpha, hill_values, shading="auto")
plt.xlabel(r'user-specified log-variance value in BurrXII, $\nu$', fontsize = 14)
plt.ylabel(r'user-specified tail exponent value in BurrXII, $\alpha = ck$', fontsize = 14)

cbar = plt.colorbar(pcm, pad = 0.01)
cbar.set_label('Estimated Hill exponent')

# --- Add text annotations ---
for i in range(len(alpha_grid)):
    for j in range(len(nu_grid)):
        plt.text(
            nu_grid[j],                 # x (nu)
            alpha_grid[i],              # y (alpha)
            f"({logvariance_values[i, j]:.2f}, \n{hill_values[i, j]:.2f})",
            ha='center',
            va='center',
            fontsize=10,
            color='white'
        )
plt.title(f"Estimated (log-variance, Hill exponent) of Burr in-degrees averaged over {num_reps_in} experiments",fontsize=16)
plt.xticks(fontsize=12)
plt.yticks(fontsize=12)
plt.tight_layout()
plt.show()

# Step 2: Confirm that the hill exponents of in- and out-degrees after Configuration Model are sensible

In [ ]:
in_deg_hill_storer = []
out_deg_hill_storer = []

for _ in tqdm(range(100)):

    k_outdeg, c_outdeg, lam_outdeg = get_parameters_for_burr_distribution(torch.tensor([2.0]), torch.tensor([10.0]), target_average_connection)
    k_indeg, c_indeg, lam_indeg = get_parameters_for_burr_distribution(torch.tensor([3.0]), torch.tensor([3.0]), target_average_connection)

    out_deg_seq_burr = generate_burr_degrees(
        c_val=c_outdeg,
        d_val=k_outdeg,
        lam_val=lam_outdeg,
        max_cutoff=burr_max_outdeg,
        num_of_firms=num_firms,
    )
    while (
        (out_deg_seq_burr.max().item() < 0.1 * num_firms)
        or (out_deg_seq_burr[out_deg_seq_burr > 0].log().var().item() < 2.95)
        or (out_deg_seq_burr[out_deg_seq_burr > 0].log().var().item() > 3.05)
        or (
            compute_hill_exponent(
                out_deg_seq_burr,
                network_config.threshold_percentile_for_hill_deg,
                is_degree=True,
            ).item()
            > 1.5
        )
        or (
            compute_hill_exponent(
                out_deg_seq_burr,
                network_config.threshold_percentile_for_hill_deg,
                is_degree=True,
            ).item()
            < 1.4
        )
    ):
        out_deg_seq_burr = generate_burr_degrees(
            c_val=c_outdeg,
            d_val=k_outdeg,
            lam_val=lam_outdeg,
            max_cutoff=burr_max_outdeg,
            num_of_firms=num_firms,
        )
        
        
    in_deg_seq_burr = generate_burr_degrees_matching_sum(
        c_val=c_indeg,
        d_val=k_indeg,
        lam_val=lam_indeg,
        max_cutoff=burr_max_indeg,
        num_of_firms=num_firms,
        degree_sum=out_deg_seq_burr.sum(),
    )
    while (
        (in_deg_seq_burr.min().item() < 0)
        or (in_deg_seq_burr[in_deg_seq_burr > 0].log().var().item() < 1.95)
        or (in_deg_seq_burr[in_deg_seq_burr > 0].log().var().item() > 2.05)
        or (
            compute_hill_exponent(
                in_deg_seq_burr,
                network_config.threshold_percentile_for_hill_deg,
                is_degree=True,
            ).item()
            < 2.3
        )
        or (
        compute_hill_exponent(
            in_deg_seq_burr,
            network_config.threshold_percentile_for_hill_deg,
            is_degree=True,
        ).item()
        > 2.4
        )
    ):
        in_deg_seq_burr = generate_burr_degrees_matching_sum(
            c_val=c_indeg,
            d_val=k_indeg,
            lam_val=lam_indeg,
            max_cutoff=burr_max_indeg,
            num_of_firms=num_firms,
            degree_sum=out_deg_seq_burr.sum(),
        )
        
    test_reordered_out_deg_seq_burr = triangular_wedge_matching(
        in_deg_seq=in_deg_seq_burr,
        out_deg_seq=out_deg_seq_burr,
        noise_strength=network_config.noise_amount_for_reordering_burrkiko,
    )

    adj_mat_init = directed_configuration_model(
        in_deg_seq_burr, test_reordered_out_deg_seq_burr
    )
    in_deg_seq_init, out_deg_seq_init = compute_degrees_from_model(adj_mat_init)

    in_deg_hill_storer.append(compute_hill_exponent(
        in_deg_seq_init, network_config.threshold_percentile_for_hill_deg, is_degree=True).item()
                            )
    out_deg_hill_storer.append(compute_hill_exponent(
        out_deg_seq_init, network_config.threshold_percentile_for_hill_deg, is_degree=True).item()
                            )

in_deg_hill_storer = np.array(in_deg_hill_storer)
out_deg_hill_storer = np.array(out_deg_hill_storer)

in_deg_hill_storer.mean(), in_deg_hill_storer.std(),out_deg_hill_storer.mean(), out_deg_hill_storer.std()

# Step 3: Confirm that engineered in- and out Burr degree sequences at hill001 coincide with pltfit

In [ ]:
out_deg_hill_list = []
in_deg_hill_list = []
out_deg_cns_list = []
in_deg_cns_list = []

target_average_connection = torch.tensor([40])

k_outdeg, c_outdeg, lam_outdeg = get_parameters_for_burr_distribution(torch.tensor([2.0]), torch.tensor([10.0]), target_average_connection)
k_indeg, c_indeg, lam_indeg = get_parameters_for_burr_distribution(torch.tensor([3.0]), torch.tensor([3.0]), target_average_connection)

for _ in tqdm(range(100)):
    out_deg_seq_burr = generate_burr_degrees(c_val=c_outdeg,
                                             d_val=k_outdeg,
                                             lam_val=lam_outdeg,
                                             max_cutoff=burr_max_outdeg,
                                             num_of_firms=100_000,
                                            )
    
    out_deg_hill_list.append(
        compute_hill_exponent(
            out_deg_seq_burr,
            network_config.threshold_percentile_for_hill_deg,
            is_degree=True,
        ).item()
    )

    in_deg_seq_burr = generate_burr_degrees_matching_sum(
        c_val=c_indeg,
        d_val=k_indeg,
        lam_val=lam_indeg,
        max_cutoff=burr_max_indeg,
        num_of_firms=100_000,
        degree_sum=out_deg_seq_burr.sum(),
    )

    in_deg_hill_list.append(
        compute_hill_exponent(
            in_deg_seq_burr,
            network_config.threshold_percentile_for_hill_deg,
            is_degree=True,
        ).item()
    )

    in_deg_cns_list.append(ig.power_law_fit(in_deg_seq_burr).alpha - 1)
    out_deg_cns_list.append(ig.power_law_fit(out_deg_seq_burr).alpha - 1)

    del (
        out_deg_seq_burr,
        in_deg_seq_burr,
    )
    gc.collect()

out_deg_hill_list = np.array(out_deg_hill_list)
in_deg_hill_list = np.array(in_deg_hill_list)
out_deg_cns_list = np.array(out_deg_cns_list)
in_deg_cns_list = np.array(in_deg_cns_list) 

In [ ]:
def _summary_row(arr):
    a = np.asarray(arr)
    return {
        "mean": float(a.mean()),
        "median": float(np.median(a)),
        "std": float(a.std()),
        "min": float(a.min()),
        "max": float(a.max()),
    }


stats_df = pd.DataFrame(
    {
        "hill_in": _summary_row(in_deg_hill_list),
        "hill_out": _summary_row(out_deg_hill_list),
        "plfit_in": _summary_row(in_deg_cns_list),
        "plfit_out": _summary_row(out_deg_cns_list),
    }
).T

fig, (ax_top, ax_bottom) = plt.subplots(
    2,
    1,
    figsize=(14, 8),
    constrained_layout=True,
    gridspec_kw={"height_ratios": [3, 1.4]},
)

ax_top.hist(in_deg_hill_list, bins=100, label="hill_in", color="blue")
ax_top.hist(out_deg_hill_list, bins=100, label="hill_out", color="red")
ax_top.hist(in_deg_cns_list, bins=100, alpha=0.5, label="plfit_in", color="green")
ax_top.hist(out_deg_cns_list, bins=100, alpha=0.5, label="plfit_out", color="orange")
ax_top.legend(fontsize = 18)
ax_top.tick_params(axis="both", labelsize=18)
ax_top.set_title(
    "Histogram of tail exponents from 100 in- and out Burr degree sequences measured by Hill estimator and plfit", fontsize = 18
)

ax_bottom.axis("off")
cell_text = [[f"{v:.3g}" for v in row] for row in stats_df.values]
tbl = ax_bottom.table(
    cellText=cell_text,
    colLabels=stats_df.columns.tolist(),
    rowLabels=stats_df.index.tolist(),
    loc="center",
    cellLoc="center",
)
tbl.auto_set_font_size(False)
tbl.scale(0.4, 2) 
tbl.set_fontsize(18)

plt.show()

# Step 4: Run 100 trials for best optuna parameters with ```runner_for_optuna_multiple_times.ipynb```

# Step 5: Run the files for Overleaf figures with ```runner_for_overleaf_figures.ipynb```

# Step 6: Ablation studies

The next cell is for integerized, truncated, reconciled Burr degrees with 100k firms at hill99\%, using $(\alpha,\nu)=(2,10)$ for out-degrees, and using $(\alpha,\nu)=(3,3)$ for in-degrees.

In [ ]:
# change this value to 1_000, 10_000, 100_000 for different network sizes in the ablation study
num_firms = 10_000
# change this value to 40 depending on the particular ablation study
target_average_connection = torch.tensor([20])

out_deg_mean_list = []
in_deg_mean_list = []
out_deg_var_list = []
in_deg_var_list = []
out_deg_max_list = []
in_deg_max_list = []
out_deg_hill_list = []
in_deg_hill_list = []
out_deg_hill_zero_list = []
in_deg_hill_zero_list = []

in_deg_wo_reconciliation_mean_list = []
in_deg_wo_reconciliation_hill_list = []
in_deg_wo_reconciliation_var_list = []
in_deg_wo_reconciliation_max_list = []

for _ in tqdm(range(5000)):
    k_outdeg, c_outdeg, lam_outdeg = get_parameters_for_burr_distribution(torch.tensor([2]), torch.tensor([10]), target_average_connection)
    k_indeg, c_indeg, lam_indeg = get_parameters_for_burr_distribution(torch.tensor([3]), torch.tensor([3]), target_average_connection)
    out_deg_seq_burr = generate_burr_degrees(c_val = c_outdeg, 
                                             d_val = k_outdeg, 
                                             lam_val = lam_outdeg, 
                                             max_cutoff = burr_max_outdeg, 
                                             num_of_firms = num_firms)
    in_deg_seq_burr = generate_burr_degrees_matching_sum(c_val = c_indeg, 
                                                 d_val = k_indeg, 
                                                 lam_val = lam_indeg, 
                                                 max_cutoff = burr_max_indeg,
                                                 num_of_firms = num_firms, 
                                                 degree_sum = out_deg_seq_burr.sum())
    in_deg_seq_burr_wo_reconciliation = generate_burr_degrees(c_val = c_indeg, 
                                                              d_val = k_indeg, 
                                                              lam_val = lam_indeg, 
                                                              max_cutoff = burr_max_indeg, 
                                                              num_of_firms = num_firms)
    
    out_deg_var_list.append(out_deg_seq_burr[out_deg_seq_burr > 0].log().var().item())
    in_deg_var_list.append(in_deg_seq_burr[in_deg_seq_burr > 0].log().var().item())
    in_deg_wo_reconciliation_var_list.append(in_deg_seq_burr_wo_reconciliation[in_deg_seq_burr_wo_reconciliation > 0].log().var().item())

    out_deg_hill_value = compute_hill_exponent(out_deg_seq_burr, 0.99, is_degree = True).item()
    in_deg_hill_value = compute_hill_exponent(in_deg_seq_burr, 0.99, is_degree = True).item()
    out_deg_hill_list.append(out_deg_hill_value)
    in_deg_hill_list.append(in_deg_hill_value)
    out_deg_hill_zero_list.append( out_deg_hill_value - k_outdeg*c_outdeg*(1 - 0.01**(1/k_outdeg)) )
    in_deg_hill_zero_list.append( in_deg_hill_value - k_indeg*c_indeg*(1 - 0.01**(1/k_indeg)) )
    in_deg_wo_reconciliation_hill_list.append(compute_hill_exponent(in_deg_seq_burr_wo_reconciliation, 0.99, is_degree = True).item())

    out_deg_max_list.append(out_deg_seq_burr.max().item())
    in_deg_max_list.append(in_deg_seq_burr.max().item())
    in_deg_wo_reconciliation_max_list.append(in_deg_seq_burr_wo_reconciliation.max().item())

    out_deg_mean_list.append(out_deg_seq_burr.float().mean().item())
    in_deg_mean_list.append(in_deg_seq_burr.float().mean().item())
    in_deg_wo_reconciliation_mean_list.append(in_deg_seq_burr_wo_reconciliation.float().mean().item())

out_deg_mean_list = np.array(out_deg_mean_list)
in_deg_mean_list = np.array(in_deg_mean_list)
out_deg_var_list = np.array(out_deg_var_list)
in_deg_var_list = np.array(in_deg_var_list)
out_deg_hill_list = np.array(out_deg_hill_list)
in_deg_hill_list = np.array(in_deg_hill_list)
out_deg_max_list = np.array(out_deg_max_list)
in_deg_max_list = np.array(in_deg_max_list)
out_deg_hill_zero_list = np.array(out_deg_hill_zero_list)
in_deg_hill_zero_list = np.array(in_deg_hill_zero_list)

in_deg_wo_reconciliation_mean_list = np.array(in_deg_wo_reconciliation_mean_list)
in_deg_wo_reconciliation_hill_list = np.array(in_deg_wo_reconciliation_hill_list)
in_deg_wo_reconciliation_var_list = np.array(in_deg_wo_reconciliation_var_list)
in_deg_wo_reconciliation_max_list = np.array(in_deg_wo_reconciliation_max_list)

print(f"out mean mean:{out_deg_mean_list.mean()}, out mean std:{out_deg_mean_list.std()}, out mean max:{out_deg_mean_list.max()}, out mean min:{out_deg_mean_list.min()}")
print(f"in mean mean:{in_deg_mean_list.mean()}, in mean std:{in_deg_mean_list.std()}, in mean max:{in_deg_mean_list.max()}, in mean min:{in_deg_mean_list.min()}")
print(f"out hill mean:{out_deg_hill_list.mean()}, out hill std:{out_deg_hill_list.std()}, out hill max:{out_deg_hill_list.max()}, out hill min:{out_deg_hill_list.min()}")
print(f"in hill mean:{in_deg_hill_list.mean()}, in hill std:{in_deg_hill_list.std()}, in hill max:{in_deg_hill_list.max()}, in hill min:{in_deg_hill_list.min()}")
print(f"out var mean:{out_deg_var_list.mean()}, out var std:{out_deg_var_list.std()}, out var max:{out_deg_var_list.max()}, out var min:{out_deg_var_list.min()}")
print(f"in var mean:{in_deg_var_list.mean()}, in var std:{in_deg_var_list.std()}, in var max:{in_deg_var_list.max()}, in var min:{in_deg_var_list.min()}")
print(f"out max mean:{out_deg_max_list.mean()}, out max std:{out_deg_max_list.std()}, out max max:{out_deg_max_list.max()}, out max min:{out_deg_max_list.min()}")
print(f"in max mean:{in_deg_max_list.mean()}, in max std:{in_deg_max_list.std()}, in max max:{in_deg_max_list.max()}, in max min:{in_deg_max_list.min()}")

In [ ]:
print(f"in hill mean:{in_deg_wo_reconciliation_hill_list.mean()}, in hill std:{in_deg_wo_reconciliation_hill_list.std()}, in hill max:{in_deg_wo_reconciliation_hill_list.max()}, in hill min:{in_deg_wo_reconciliation_hill_list.min()}")
print(f"in var mean:{in_deg_wo_reconciliation_var_list.mean()}, in var std:{in_deg_wo_reconciliation_var_list.std()}, in var max:{in_deg_wo_reconciliation_var_list.max()}, in var min:{in_deg_wo_reconciliation_var_list.min()}")
print(f"in max mean:{in_deg_wo_reconciliation_max_list.mean()}, in max std:{in_deg_wo_reconciliation_max_list.std()}, in max max:{in_deg_wo_reconciliation_max_list.max()}, in max min:{in_deg_wo_reconciliation_max_list.min()}")
print(f"in mean mean:{in_deg_wo_reconciliation_mean_list.mean()}, in mean std:{in_deg_wo_reconciliation_mean_list.std()}, in mean max:{in_deg_wo_reconciliation_mean_list.max()}, in mean min:{in_deg_wo_reconciliation_mean_list.min()}")

In [ ]:
print(f"out mean:{out_deg_hill_zero_list.mean()}, out std:{out_deg_hill_zero_list.std()}, out max:{out_deg_hill_zero_list.max()}, out min:{out_deg_hill_zero_list.min()}")
print(f"in mean:{in_deg_hill_zero_list.mean()}, in std:{in_deg_hill_zero_list.std()}, in max:{in_deg_hill_zero_list.max()}, in min:{in_deg_hill_zero_list.min()}")

## just integerization, at hill99%

In [ ]:
from scipy.stats import burr12

# change this value to 1_000, 10_000, 100_000 for different network sizes in the ablation study
num_firms = 10_000
# change this value to 40 depending on the particular ablation study
target_average_connection = torch.tensor([20])

out_deg_mean_list = []
in_deg_mean_list = []
out_deg_var_list = []
in_deg_var_list = []
out_deg_max_list = []
in_deg_max_list = []
out_deg_hill_list = []
in_deg_hill_list = []
out_deg_hill_zero_list = []
in_deg_hill_zero_list = []

for _ in tqdm(range(5000)):
    k_outdeg, c_outdeg, lam_outdeg = get_parameters_for_burr_distribution(torch.tensor([2]), torch.tensor([10]), target_average_connection)
    k_indeg, c_indeg, lam_indeg = get_parameters_for_burr_distribution(torch.tensor([3]), torch.tensor([3]), target_average_connection)
    out_deg_seq_burr = torch.tensor(burr12.rvs(c=c_outdeg, d=k_outdeg, scale=lam_outdeg, size=num_firms)).round()
    in_deg_seq_burr = torch.tensor(burr12.rvs(c=c_indeg, d=k_indeg, scale=lam_indeg, size=num_firms)).round()

    out_deg_var_list.append(out_deg_seq_burr[out_deg_seq_burr > 0].log().var().item())
    in_deg_var_list.append(in_deg_seq_burr[in_deg_seq_burr > 0].log().var().item())

    out_deg_hill_value = compute_hill_exponent(out_deg_seq_burr, 0.99, is_degree = True).item()
    in_deg_hill_value = compute_hill_exponent(in_deg_seq_burr, 0.99, is_degree = True).item()
    out_deg_hill_list.append(out_deg_hill_value)
    in_deg_hill_list.append(in_deg_hill_value)
    out_deg_hill_zero_list.append( out_deg_hill_value - k_outdeg*c_outdeg*(1 - 0.01**(1/k_outdeg)) )
    in_deg_hill_zero_list.append( in_deg_hill_value - k_indeg*c_indeg*(1 - 0.01**(1/k_indeg)) )
    
    out_deg_max_list.append(out_deg_seq_burr.max().item())
    in_deg_max_list.append(in_deg_seq_burr.max().item())

    out_deg_mean_list.append(out_deg_seq_burr.mean().item())
    in_deg_mean_list.append(in_deg_seq_burr.mean().item())

out_deg_mean_list = np.array(out_deg_mean_list)
in_deg_mean_list = np.array(in_deg_mean_list)
out_deg_var_list = np.array(out_deg_var_list)
in_deg_var_list = np.array(in_deg_var_list)
out_deg_hill_list = np.array(out_deg_hill_list)
in_deg_hill_list = np.array(in_deg_hill_list)
out_deg_max_list = np.array(out_deg_max_list)
in_deg_max_list = np.array(in_deg_max_list)
out_deg_hill_zero_list = np.array(out_deg_hill_zero_list)
in_deg_hill_zero_list = np.array(in_deg_hill_zero_list)

print(f"out mean mean:{out_deg_mean_list.mean()}, out mean std:{out_deg_mean_list.std()}, out mean max:{out_deg_mean_list.max()}, out mean min:{out_deg_mean_list.min()}")
print(f"in mean mean:{in_deg_mean_list.mean()}, in mean std:{in_deg_mean_list.std()}, in mean max:{in_deg_mean_list.max()}, in mean min:{in_deg_mean_list.min()}")
print(f"out hill mean:{out_deg_hill_list.mean()}, out hill std:{out_deg_hill_list.std()}, out hill max:{out_deg_hill_list.max()}, out hill min:{out_deg_hill_list.min()}")
print(f"in hill mean:{in_deg_hill_list.mean()}, in hill std:{in_deg_hill_list.std()}, in hill max:{in_deg_hill_list.max()}, in hill min:{in_deg_hill_list.min()}")
print(f"out var mean:{out_deg_var_list.mean()}, out var std:{out_deg_var_list.std()}, out var max:{out_deg_var_list.max()}, out var min:{out_deg_var_list.min()}")
print(f"in var mean:{in_deg_var_list.mean()}, in var std:{in_deg_var_list.std()}, in var max:{in_deg_var_list.max()}, in var min:{in_deg_var_list.min()}")
print(f"out max mean:{out_deg_max_list.mean()}, out max std:{out_deg_max_list.std()}, out max max:{out_deg_max_list.max()}, out max min:{out_deg_max_list.min()}")
print(f"in max mean:{in_deg_max_list.mean()}, in max std:{in_deg_max_list.std()}, in max max:{in_deg_max_list.max()}, in max min:{in_deg_max_list.min()}")

In [ ]:
print(f"out mean:{out_deg_hill_zero_list.mean()}, out std:{out_deg_hill_zero_list.std()}, out max:{out_deg_hill_zero_list.max()}, out min:{out_deg_hill_zero_list.min()}")
print(f"in mean:{in_deg_hill_zero_list.mean()}, in std:{in_deg_hill_zero_list.std()}, in max:{in_deg_hill_zero_list.max()}, in min:{in_deg_hill_zero_list.min()}")

# none, at hill99%

In [ ]:
# change this value to 1_000, 10_000, 100_000 for different network sizes in the ablation study
num_firms = 10_000
# change this value to 40 depending on the particular ablation study
target_average_connection = torch.tensor([20])

out_deg_mean_list = []
in_deg_mean_list = []
out_deg_var_list = []
in_deg_var_list = []
out_deg_max_list = []
in_deg_max_list = []
out_deg_hill_list = []
in_deg_hill_list = []
out_deg_hill_zero_list = []
in_deg_hill_zero_list = []

for _ in tqdm(range(5000)):
    k_outdeg, c_outdeg, lam_outdeg = get_parameters_for_burr_distribution(torch.tensor([2]), torch.tensor([10]), target_average_connection)
    k_indeg, c_indeg, lam_indeg = get_parameters_for_burr_distribution(torch.tensor([3]), torch.tensor([3]), target_average_connection)
    out_deg_seq_burr = torch.tensor(burr12.rvs(c=c_outdeg, d=k_outdeg, scale=lam_outdeg, size=num_firms))
    in_deg_seq_burr = torch.tensor(burr12.rvs(c=c_indeg, d=k_indeg, scale=lam_indeg, size=num_firms))

    out_deg_var_list.append(out_deg_seq_burr[out_deg_seq_burr > 0].log().var().item())
    in_deg_var_list.append(in_deg_seq_burr[in_deg_seq_burr > 0].log().var().item())

    out_deg_hill_value = compute_hill_exponent(out_deg_seq_burr, 0.99, is_degree = False).item()
    in_deg_hill_value = compute_hill_exponent(in_deg_seq_burr, 0.99, is_degree = False).item()
    out_deg_hill_list.append(out_deg_hill_value)
    in_deg_hill_list.append(in_deg_hill_value)
    out_deg_hill_zero_list.append( out_deg_hill_value - k_outdeg*c_outdeg*(1 - 0.01**(1/k_outdeg)) )
    in_deg_hill_zero_list.append( in_deg_hill_value - k_indeg*c_indeg*(1 - 0.01**(1/k_indeg)) )
    
    out_deg_max_list.append(out_deg_seq_burr.max().item())
    in_deg_max_list.append(in_deg_seq_burr.max().item())

    out_deg_mean_list.append(out_deg_seq_burr.mean().item())
    in_deg_mean_list.append(in_deg_seq_burr.mean().item())

out_deg_mean_list = np.array(out_deg_mean_list)
in_deg_mean_list = np.array(in_deg_mean_list)
out_deg_var_list = np.array(out_deg_var_list)
in_deg_var_list = np.array(in_deg_var_list)
out_deg_hill_list = np.array(out_deg_hill_list)
in_deg_hill_list = np.array(in_deg_hill_list)
out_deg_max_list = np.array(out_deg_max_list)
in_deg_max_list = np.array(in_deg_max_list)
out_deg_hill_zero_list = np.array(out_deg_hill_zero_list)
in_deg_hill_zero_list = np.array(in_deg_hill_zero_list)

print(f"out mean mean:{out_deg_mean_list.mean()}, out mean std:{out_deg_mean_list.std()}, out mean max:{out_deg_mean_list.max()}, out mean min:{out_deg_mean_list.min()}")
print(f"in mean mean:{in_deg_mean_list.mean()}, in mean std:{in_deg_mean_list.std()}, in mean max:{in_deg_mean_list.max()}, in mean min:{in_deg_mean_list.min()}")
print(f"out hill mean:{out_deg_hill_list.mean()}, out hill std:{out_deg_hill_list.std()}, out hill max:{out_deg_hill_list.max()}, out hill min:{out_deg_hill_list.min()}")
print(f"in hill mean:{in_deg_hill_list.mean()}, in hill std:{in_deg_hill_list.std()}, in hill max:{in_deg_hill_list.max()}, in hill min:{in_deg_hill_list.min()}")
print(f"out var mean:{out_deg_var_list.mean()}, out var std:{out_deg_var_list.std()}, out var max:{out_deg_var_list.max()}, out var min:{out_deg_var_list.min()}")
print(f"in var mean:{in_deg_var_list.mean()}, in var std:{in_deg_var_list.std()}, in var max:{in_deg_var_list.max()}, in var min:{in_deg_var_list.min()}")
print(f"out max mean:{out_deg_max_list.mean()}, out max std:{out_deg_max_list.std()}, out max max:{out_deg_max_list.max()}, out max min:{out_deg_max_list.min()}")
print(f"in max mean:{in_deg_max_list.mean()}, in max std:{in_deg_max_list.std()}, in max max:{in_deg_max_list.max()}, in max min:{in_deg_max_list.min()}")

In [ ]:
print(f"out mean:{out_deg_hill_zero_list.mean()}, out std:{out_deg_hill_zero_list.std()}, out max:{out_deg_hill_zero_list.max()}, out min:{out_deg_hill_zero_list.min()}")
print(f"in mean:{in_deg_hill_zero_list.mean()}, in std:{in_deg_hill_zero_list.std()}, in max:{in_deg_hill_zero_list.max()}, in min:{in_deg_hill_zero_list.min()}")